In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

btc = pd.read_csv("../data_clean/btc_clean.csv", index_col="date", parse_dates=True)
p, r = btc["price"], btc["ret"]

df = pd.DataFrame({
    "y":   r.shift(-1),              # tomorrow's return (the thing we predict)
    "mom": p / p.shift(30) - 1,      # 30-day momentum, known today
    "vol": r.rolling(30).std(),      # 30-day volatility, known today
})
train = df.loc["2018":"2021"].dropna()
print(len(train))

1461


In [2]:
X = sm.add_constant(train[["mom", "vol"]])     # adds the intercept a
model = sm.OLS(train["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 30})
print(model.params.round(5))
print(model.tvalues.round(2))
print(model.rsquared)

const    0.00029
mom      0.00194
vol      0.03414
dtype: float64
const    0.09
mom      0.46
vol      0.43
dtype: float64
0.00026021003869713955


In [3]:
data = df.loc["2018":"2025-12-30"].dropna()
rows = []
for t in data.loc["2022":].index:
    past = data.loc[:t].iloc[:-1]       # rows before today: their "tomorrow" is already known
    m = sm.OLS(past["y"], sm.add_constant(past[["mom", "vol"]])).fit()
    x_today = [1, data.loc[t, "mom"], data.loc[t, "vol"]]
    rows.append({
        "date": t,
        "model": np.dot(m.params.values, x_today),   # momentum forecast for tomorrow
        "bench": r.loc["2018":t].mean(),             # benchmark: average return so far
        "actual": data.loc[t, "y"],                  # what really happened tomorrow
    })
oos = pd.DataFrame(rows).set_index("date")
print(len(oos))

1460


In [4]:
def score(d):
    em, eb = d["actual"] - d["model"], d["actual"] - d["bench"]      # forecast errors
    r2 = 1 - (em**2).sum() / (eb**2).sum()
    f = eb**2 - (em**2 - (d["bench"] - d["model"])**2)               # Clark–West adjustment
    cw = sm.OLS(f, np.ones(len(f))).fit(cov_type="HAC", cov_kwds={"maxlags": 30})
    return round(r2 * 100, 3), round(cw.tvalues.iloc[0], 2)

print("validation 2022-23:", score(oos.loc["2022":"2023"]))
print("test 2024-25:      ", score(oos.loc["2024":"2025"]))

validation 2022-23: (np.float64(0.009), np.float64(0.42))
test 2024-25:       (np.float64(-0.017), np.float64(0.19))


In [5]:
oos.to_csv("../outputs/momentum_oos.csv")